# Notebook 03 — Real Embeddings (384 dimensions, free & local)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

So far: hand-built 4-D vectors — great for intuition, useless for real text. Now we use a
real model, **all-MiniLM-L6-v2** from HuggingFace, free and running on your own machine. It
turns each sentence into a **384-dimensional** vector. Every tool from Notebooks 01–02
(cosine similarity, nearest neighbours, PCA) works unchanged. Only the number of directions
grows: 4 → 384.

**Cost:** zero. No account, no API key. The model runs locally.

In [ ]:
%pip install -q numpy matplotlib scikit-learn sentence-transformers
print("Ready.")

## Step 1 — Load the model

> HuggingFace is like a public library of trained models: you borrow one for free, no
> sign-up. The first run downloads MiniLM (~80 MB) once, then loads instantly after.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded. It turns any text into a 384-number vector.")

## Step 2 — Twelve sentences

Four sentences each from three obvious groups — **animals**, **vehicles**, **food**. The
model never sees these labels; we'll check whether it groups them anyway.

In [ ]:
sentences = [
    # Animals
    "The dog wagged its tail when its owner came home.",
    "A kitten chased a ball of yarn across the floor.",
    "Lions live in prides on the African savanna.",
    "The parrot mimicked every word the children said.",
    # Vehicles
    "I drove the car to the grocery store this morning.",
    "The new electric bicycle has a range of sixty miles.",
    "Trucks deliver packages to our neighbourhood every day.",
    "The airplane landed smoothly despite the strong winds.",
    # Food
    "She baked sourdough bread for the first time on Sunday.",
    "The pizza was hot and covered in melted mozzarella.",
    "I ordered sushi for lunch at the new Japanese restaurant.",
    "He grilled steak and roasted vegetables for dinner.",
]
categories = ["animal"] * 4 + ["vehicle"] * 4 + ["food"] * 4
colour_map = {"animal": "tab:orange", "vehicle": "tab:blue", "food": "tab:red"}

print(f"{len(sentences)} sentences, {len(set(categories))} groups.")

## Step 3 — Embed all twelve at once

`model.encode(list_of_sentences)` returns one vector per sentence, as a NumPy array of shape
(12, 384) — twelve sentences, each 384 numbers. Compare with Notebook 01: ten words, each 4
numbers. **Same idea, more dials.**

In [ ]:
embeddings = model.encode(sentences)

print(f"Shape   : {embeddings.shape}   (sentences, directions)")
print(f"Type    : {embeddings.dtype}")
print(f"First 5 of sentence 0: {embeddings[0, :5]}")
print("\nEmbedded. Each sentence is now 384 numbers.")

### What to notice
- **(12, 384)** — twelve sentences, each a 384-number vector.
- The numbers look like noise to us, but they encode meaning. We read them through
  *similarity scores* and *plots*, never directly.
- Twelve sentences is tiny. Picture a million: 1,000,000 × 384 numbers. Storing and
  searching that is the job of a vector store — Notebook 04.

## Step 4 — Cosine similarity, two ways

First with **our own** function from Notebook 01 (proof the idea didn't change), then the
fast NumPy way for all pairs at once.

In [ ]:
# Activity: our own cosine from Notebook 01 (works on a vector of any length).
import math

def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

def magnitude(v):
    return math.sqrt(sum(x * x for x in v))

def cosine_similarity(a, b):
    return dot_product(a, b) / (magnitude(a) * magnitude(b))

dog, kitten = embeddings[0], embeddings[1]
print(f"our cosine(dog, kitten) = {cosine_similarity(dog, kitten):.3f}")

Now the fast library way — all 12x12 pairs in one matrix multiply — and check it
gives the same number.

In [ ]:
# Activity: the library way (normalize every row, then one matrix multiply).
norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
sim_matrix = norm @ norm.T

print(f"library cosine(dog, kitten)  = {sim_matrix[0, 1]:.3f}   (same number)")
print(f"cosine(dog, dog)             = {sim_matrix[0, 0]:.3f}   (a thing with itself = 1.0)")
print(f"cosine(dog, sourdough bread) = {sim_matrix[0, 8]:.3f}   (different topics = lower)")

## Words vs sentences — and how context steers a word

A single word is just a very short text, so the model embeds it like anything else, and you
can compare a word against a sentence with the same cosine. Then the famous "bank" test:
two sentences that BOTH contain "bank", probed for water-meaning and money-meaning. Watch
context steer each one.

In [ ]:
# Activity: compare a word to a word, and a word to a sentence.
w_puppy = model.encode("puppy")
w_dog = model.encode("dog")
w_car = model.encode("car")

print("word vs word:")
print("  puppy vs dog:", round(cosine_similarity(w_puppy, w_dog), 3))   # ~0.804 close
print("  puppy vs car:", round(cosine_similarity(w_puppy, w_car), 3))   # ~0.464 far

print("word vs sentence:")
print("  dog vs dog sentence  :", round(cosine_similarity(w_dog, embeddings[0]), 3))  # ~0.470
print("  dog vs pizza sentence:", round(cosine_similarity(w_dog, embeddings[9]), 3))  # ~0.10

In [ ]:
# Activity: the "bank" test — context steers the same word two different ways.
A = model.encode("We sat on the river bank and watched the water.")
B = model.encode("I deposited my salary at the bank this morning.")
water = model.encode("a river with flowing water")
money = model.encode("money and finance")

print("A vs B (both contain 'bank'):", round(cosine_similarity(A, B), 3))   # ~0.281 NOT close!
print("A vs water probe:", round(cosine_similarity(A, water), 3))           # ~0.531 high
print("B vs water probe:", round(cosine_similarity(B, water), 3))           # ~0.098 low
print("A vs money probe:", round(cosine_similarity(A, money), 3))           # ~0.190 low
print("B vs money probe:", round(cosine_similarity(B, money), 3))           # ~0.301 higher

## Know your model: dimension, max length, size

Three facts to know about any embedding model. For ours: dimension **384**, max sequence
length **256 tokens** (a token is a word or word-piece — "wagged" reads as `wa` + `gged`;
256 tokens is roughly 190-200 words), size **~80 MB**. The sharp edge: text past the limit
is **silently cut off**. Prove it:

In [ ]:
# Activity: read the model's own fact sheet.
print("max sequence length:", model.max_seq_length, "tokens")
print("dimension:", model.get_sentence_embedding_dimension())
print("'wagged' becomes the tokens:", model.tokenizer.tokenize("wagged"))

In [ ]:
# Activity: the truncation trap — text past 256 tokens is silently ignored.
filler = "The committee reviewed the quarterly schedule and noted the agenda. " * 40

probe = model.encode("pizza and italian food")
at_end   = model.encode(filler + " The secret topic of this document is pizza.")
at_front = model.encode("The secret topic of this document is pizza. " + filler)

print("pizza at the END  :", round(cosine_similarity(probe, at_end), 3))    # ~0.102 never read
print("pizza at the FRONT:", round(cosine_similarity(probe, at_front), 3))  # ~0.231 read, diluted
# (You may see a tokenizer warning about the text being too long — that's the point!)

## Step 5 — Predict, then verify: three searches

For each query, guess which of the twelve sentences is **closest** in meaning before you run
it. Note: the queries share almost no words with their best matches — meaning is what counts,
not shared words.

- Q1: "a pet that climbs trees and purrs"
- Q2: "how do I travel between cities?"
- Q3: "what should I cook for guests tonight?"

In [ ]:
queries = [
    "a pet that climbs trees and purrs",
    "how do I travel between cities?",
    "what should I cook for guests tonight?",
]
query_vecs = model.encode(queries)

for q, qv in zip(queries, query_vecs):
    sims = np.array([cosine_similarity(qv, ev) for ev in embeddings])
    top = int(np.argmax(sims))
    bottom = int(np.argmin(sims))
    print(f"\nQuery: {q!r}")
    print(f"  CLOSEST  [{sims[top]:.3f}]  {sentences[top]!r}")
    print(f"  FARTHEST [{sims[bottom]:.3f}]  {sentences[bottom]!r}")

### What to notice
- **It understands paraphrase.** "a pet that climbs trees and purrs" finds an *animal*
  sentence — even though it shares no words like "climb" or "purr" with it. (For this small
  set the top animal match is the dog sentence, not the kitten — close calls happen. The
  model deals in shades of meaning, not certainties.)
- **The top score is not 1.0.** Real matches score low — often ~0.2–0.5 here. Don't expect
  near-1.0; that only happens for nearly identical text. What matters is *which* is highest.

## Step 6 — See the twelve sentences with PCA

Same PCA as Notebook 02, now squashing **384 → 2** instead of 4 → 2. More is lost, so the
picture is rougher — but the groups should still show.

In [ ]:
pca = PCA(n_components=2)
pcs = pca.fit_transform(embeddings)
var = pca.explained_variance_ratio_

plt.figure(figsize=(10, 6))
for (x, y), cat, sent in zip(pcs, categories, sentences):
    plt.scatter(x, y, c=colour_map[cat], s=150, edgecolor="black")
    plt.annotate(sent[:25] + "…", (x, y), xytext=(6, 4),
                 textcoords="offset points", fontsize=9)
for cat, c in colour_map.items():
    plt.scatter([], [], c=c, s=150, edgecolor="black", label=cat)
plt.legend(loc="best")
plt.xlabel(f"PC1  ({var[0] * 100:.0f}% of spread)")
plt.ylabel(f"PC2  ({var[1] * 100:.0f}% of spread)")
plt.title("12 sentences in 384-D, flattened to 2-D")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

print(f"PC1 + PC2 capture {sum(var) * 100:.0f}% of the spread "
      f"(much less than the 4-D case — most meaning is in the other directions).")

## Recap

- A real model turns each sentence into a 384-number vector with one `encode` call.
- Read an embedding's shape: first number = how many items, second = the vector size.
- Our hand-written cosine and the library's give the same answer — the idea never changed.
- The model matches *meaning*, not shared words; top cosines are well below 1.0.
- A PCA plot is a rough approximation of high-D similarity. When the plot and the cosine
  disagree, trust the cosine.

**Next (Notebook 04):** we found nearest sentences by comparing against all twelve by hand.
At a million sentences that breaks down. A **vector store** (ChromaDB) does the storing and
searching for us.